In [0]:
DESCRIBE TABLE samples.tpcds_sf1000.store_sales;

col_name,data_type,comment
ss_sold_date_sk,int,null
ss_sold_time_sk,int,null
ss_item_sk,int,null
ss_customer_sk,int,null
ss_cdemo_sk,int,null
ss_hdemo_sk,int,null
ss_addr_sk,int,null
ss_store_sk,int,null
ss_promo_sk,int,null
ss_ticket_number,bigint,null


In [0]:
DESCRIBE DETAIL samples.tpcds_sf1000.store_sales;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics
delta,87632b72-90da-4a0c-90a9-5769f0764c41,samples.tpcds_sf1000.store_sales,null,s3://system-tables-prod-eu-central-1-uc-metastore-bucket/metastore/49a6a9df-5e06-47f5-8d35-b3b6ea541718/tables/e0de96c3-71b0-4caa-bada-b8bc091dcea4,2025-05-08T09:23:58.949Z,2025-05-09T21:42:13Z,List(ss_sold_date_sk),List(),1837,103631627628,Map(),1,2,"List(appendOnly, invariants)",Map()


In [0]:
DESCRIBE EXTENDED samples.nyctaxi.trips;

col_name,data_type,comment
tpep_pickup_datetime,timestamp,null
tpep_dropoff_datetime,timestamp,null
trip_distance,double,null
fare_amount,double,null
pickup_zip,int,null
dropoff_zip,int,null
,,
# Delta Statistics Columns,,
Column Names,"tpep_dropoff_datetime, trip_distance, pickup_zip, fare_amount, tpep_pickup_datetime, dropoff_zip",
Column Selection Method,first-32,


In [0]:
CREATE OR REPLACE TABLE `utm-master-an2-sd&ia-dl`.armin_andrei_chanchian.store_sales_small
USING DELTA
AS
SELECT
  ss.*,
  d.d_year
FROM samples.tpcds_sf1000.store_sales ss
JOIN samples.tpcds_sf1000.date_dim d
  ON ss.ss_sold_date_sk = d.d_date_sk
WHERE d.d_year = 2001
LIMIT 5000;

num_affected_rows,num_inserted_rows


In [0]:
SELECT COUNT(*) 
FROM `utm-master-an2-sd&ia-dl`.armin_andrei_chanchian.store_sales_small;

count(1)
5000


In [0]:
DESCRIBE DETAIL `utm-master-an2-sd&ia-dl`.armin_andrei_chanchian.store_sales_small;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics
delta,f9d5e2d9-ae10-4d45-84a3-1283a95414b9,utm-master-an2-sd&ia-dl.armin_andrei_chanchian.store_sales_small,null,s3://utm-dbx-platform-metastore-storage/metastore/1c85d2e6-5f71-4795-a58d-d0dc96124bed/tables/5e7e1f6d-3a29-4623-8c61-550c5c11ccd5,2026-01-17T10:24:22.797Z,2026-01-17T10:25:21Z,List(),List(),1,279761,Map(delta.enableDeletionVectors -> true),3,7,List(deletionVectors),"Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)"


In [0]:
CREATE OR REPLACE TABLE `utm-master-an2-sd&ia-dl`.armin_andrei_chanchian.store_sales_small_part
USING DELTA
PARTITIONED BY (d_year)
AS
SELECT *
FROM `utm-master-an2-sd&ia-dl`.armin_andrei_chanchian.store_sales_small;

num_affected_rows,num_inserted_rows


In [0]:
DESCRIBE DETAIL `utm-master-an2-sd&ia-dl`.armin_andrei_chanchian.store_sales_small_part;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics
delta,7edb4a0e-05a8-4009-a29b-db64b0d1c693,utm-master-an2-sd&ia-dl.armin_andrei_chanchian.store_sales_small_part,null,s3://utm-dbx-platform-metastore-storage/metastore/1c85d2e6-5f71-4795-a58d-d0dc96124bed/tables/abb245d3-a0a6-430e-8e36-f6de59db9e83,2026-01-17T10:25:31.638Z,2026-01-17T10:25:35Z,List(d_year),List(),1,279488,Map(delta.enableDeletionVectors -> true),3,7,List(deletionVectors),"Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)"


In [0]:
EXPLAIN
SELECT SUM(ss_net_paid)
FROM `utm-master-an2-sd&ia-dl`.armin_andrei_chanchian.store_sales_small_part
WHERE d_year = 2001;

plan
"== Physical Plan == AdaptiveSparkPlan isFinalPlan=false +- == Initial Plan == HashAggregate(keys=[], functions=[finalmerge_sum(merge sum#9112L) AS sum(UnscaledValue(ss_net_paid#9104))#9108L]) +- Exchange SinglePartition, ENSURE_REQUIREMENTS, [plan_id=4418] +- HashAggregate(keys=[], functions=[partial_sum(UnscaledValue(ss_net_paid#9104)) AS sum#9112L]) +- Project [ss_net_paid#9104] +- Filter if (isnotnull(_databricks_internal_edge_computed_column_skip_row#9244)) (_databricks_internal_edge_computed_column_skip_row#9244 = false) else isnotnull(raise_error(DELTA_SKIP_ROW_COLUMN_NOT_FILLED, map(keys: [], values: []), NullType)) +- FileScan parquet utm-master-an2-sd&ia-dl.armin_andrei_chanchian.store_sales_small_part[ss_net_paid#9104,_databricks_internal_edge_computed_column_skip_row#9244,d_year#9107] Batched: true, DataFilters: [], Format: Parquet, Location: PreparedDeltaFileIndex(1 paths)[s3://utm-dbx-platform-metastore-storage/metastore/1c85d2e6-5f71-4..., PartitionFilters: [isnotnull(d_year#9107), (d_year#9107 = 2001)], PushedFilters: [], ReadSchema: struct"


In [0]:
CREATE OR REPLACE TEMP VIEW features_ml AS
SELECT
  ss_customer_sk,
  d_year,
  SUM(ss_net_paid) AS total_spent,
  COUNT(*) AS num_transactions
FROM `utm-master-an2-sd&ia-dl`.armin_andrei_chanchian.store_sales_small_part
GROUP BY ss_customer_sk, d_year;

In [0]:
CREATE OR REPLACE TEMP VIEW features_ml AS
SELECT
  ss_customer_sk,
  d_year,
  SUM(ss_net_paid) AS total_spent,
  COUNT(*) AS num_transactions
FROM `utm-master-an2-sd&ia-dl`.armin_andrei_chanchian.store_sales_small_part
GROUP BY ss_customer_sk, d_year;

In [0]:
SELECT *
FROM `utm-master-an2-sd&ia-dl`.armin_andrei_chanchian.features_ml_table;

ss_customer_sk,d_year,total_spent,num_transactions
4658531,2001,39612.56,12
1947020,2001,21708.31,7
3966819,2001,8287.18,11
3115614,2001,17207.04,9
10342002,2001,15935.16,14
8933474,2001,7694.53,12
1070074,2001,21795.26,11
4196687,2001,17958.05,13
9042020,2001,10824.19,14
1394799,2001,20009.60,7
